In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import (
    col, round, sum, avg, count,
    rank, dense_rank, row_number,
    lag, lead, first, last,
    percent_rank, ntile, desc
)
from pyspark.sql.window import Window

schema = StructType([
    StructField("sale_id",    IntegerType(), True),
    StructField("rep",        StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("product",    StringType(),  True),
    StructField("amount",     DoubleType(),  True),
    StructField("sale_date",  StringType(),  True),
])

data = [
    (1,  "Alice", "North", "Laptop",  1200.0, "2024-01-01"),
    (2,  "Bob",   "South", "Phone",    800.0, "2024-01-01"),
    (3,  "Alice", "North", "Laptop",   950.0, "2024-01-02"),
    (4,  "Carol", "East",  "Tablet",   600.0, "2024-01-02"),
    (5,  "Bob",   "South", "Laptop",  1100.0, "2024-01-03"),
    (6,  "Alice", "North", "Phone",    750.0, "2024-01-03"),
    (7,  "Carol", "East",  "Laptop",  1300.0, "2024-01-04"),
    (8,  "Bob",   "South", "Tablet",   450.0, "2024-01-04"),
    (9,  "Alice", "North", "Tablet",   580.0, "2024-01-05"),
    (10, "Carol", "East",  "Phone",    720.0, "2024-01-05"),
    (11, "Bob",   "South", "Phone",    880.0, "2024-01-06"),
    (12, "Alice", "North", "Laptop",  1050.0, "2024-01-06"),
]

df = spark.createDataFrame(data, schema)
df.write.format("delta").mode("overwrite").saveAsTable("sales_reps")
df.show()

In [0]:
window_rep = Window.partitionBy("rep").orderBy(desc("amount"))

df.withColumn("rank", rank().over(window_rep)) \
    .withColumn("dense_rank", dense_rank().over(window_rep)) \
    .withColumn("row_number", row_number().over(window_rep)) \
    .withColumn("percent_rank", round(percent_rank().over(window_rep), 2)) \
    .select('rep', "amount", "rank", "dense_rank", "row_number", "percent_rank") \
    .sort("rep", "rank") \
    .show()

In [0]:
window_date = Window.partitionBy("rep").orderBy("sale_date")

df.withColumn("prev_amount", lag("amount", 1).over(window_date)) \
    .withColumn("next_amount", lead("amount", 1).over(window_date)) \
    .withColumn("change_from_prev", round(col("amount") - lag("amount", 1).over(window_date),2)) \
    .select("rep", "sale_date", "amount", "prev_amount", "next_amount", "change_from_prev") \
    .sort("rep", "sale_date") \
    .show()

In [0]:
window_date2 = Window.partitionBy("rep").orderBy("sale_date")

df.withColumn("prev_amount", lag("amount",1).over(window_date2)) \
    .withColumn("trend", col("amount") - lag("amount", 1).over(window_date2)) \
    .select("rep", "sale_date", "amount", "prev_amount", round(col("amount") - lag("amount",1 ).over(window_date2), 2 ).alias("trend")) \
    .sort("rep", "sale_date") \
    .show()

In [0]:
window_running = Window.partitionBy("rep").orderBy("sale_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn("running_total", round(sum("amount").over(window_running),2)) \
    .select("rep", "sale_date", "amount", "running_total") \
    .sort("rep", "sale_date") \
    .show()

In [0]:
window_moving = Window.partitionBy("rep").orderBy("sale_date").rowsBetween(-1,1)

df.withColumn("moving_avg_3", round(avg("amount").over(window_moving), 2)) \
    .select("rep", "sale_date", "amount", "moving_avg_3") \
    .sort("rep","sale_date") \
    .show()

In [0]:
window_ntile = Window.orderBy(desc("amount"))

df.withColumn("quartile", ntile(4).over(window_ntile)) \
    .select("rep", "amount", "quartile") \
    .sort("quartile", desc("amount")) \
    .show()

In [0]:
%sql
select
    rep,
    round(sum(amount),2) as total_revenue,
    round(avg(amount),2) as avg_sale,
    count(sale_id) as total_sales,
    rank() over (order by sum(amount) desc) as revenue_rank,
    round(sum(amount) / sum(sum(amount)) over() * 100, 1) as pct_of_total
from sales_reps
group by rep
order by revenue_rank

In [0]:
%sql
with rep_stats as(
  select
    rep,
    region,
    sale_date,
    amount,
    sum(amount) over(partition by rep order by sale_date rows between unbounded preceding and current row) as running_total,
    round(avg(amount) over (partition by rep order by sale_date rows between 1 preceding and 1 following), 2) as daily_rank
  from sales_reps
)

select * 
from rep_stats
order by rep, sale_date